# Beispiele: Klassifikation

Logistische Regression mit Threshold, mehr als zwei Klassen, kNN, Naive Bayes, SVM, Entscheidungsbaum und unausgewogene Klassen. Datensatz: `load_breast_cancer`, 569 Gewebeproben mit 30 Messwerten. Ziel: bösartig (0) oder gutartig (1).

- **Herkunft:** selbst erstellt für die Lernapp, kein HSLU-Kursmaterial. Die Daten sind in scikit-learn eingebaut oder werden erzeugt.
- **Ausführen:** von oben nach unten, das Notebook hängt von keinem anderen ab. Lokal in der Lernapp oder auf jedem anderen PC im Browser: [In Colab öffnen](https://colab.research.google.com/github/rotscher-baumi/lernapp-cas-ml/blob/main/notebooks/beispiele/02_Klassifikation.ipynb) (ein Google-Konto genügt, solange das Repo öffentlich ist; ist es privat, in Colab mit GitHub anmelden und «Include private repos» anhaken).
- **Navigation:** Das Inhaltsverzeichnis links in JupyterLab springt zu jedem Abschnitt. Zurück zur Übersicht: Lernapp, Kapitel «Übersicht».

In [ ]:
# Colab has scikit-learn, pandas and matplotlib; umap and imblearn are installed there on demand
import importlib.util, subprocess, sys
fehlend = [pip for mod, pip in [("umap", "umap-learn"), ("imblearn", "imbalanced-learn")] if importlib.util.find_spec(mod) is None]
if fehlend and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *fehlend], check=True)
elif fehlend:
    print("Für einzelne Beispiele fehlt: pip install " + " ".join(fehlend))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import (load_diabetes, load_breast_cancer, load_wine, load_digits,
                              make_moons, make_blobs, make_classification)
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, roc_auc_score, silhouette_score,
                             adjusted_rand_score)

SEED = 42
plt.rcParams["figure.figsize"] = (6, 3.5)
pd.set_option("display.precision", 3)

## Daten

In [ ]:
Xc, yc = load_breast_cancer(return_X_y=True, as_frame=True)
Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.25, stratify=yc, random_state=SEED)
print(Xc.shape, "Anteil gutartig:", round(yc.mean(), 2))

## Logistische Regression

Skalieren und Modell in einer Pipeline. `predict_proba` liefert die Wahrscheinlichkeit, `predict` wendet den Threshold 0,5 an.

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xc_train, yc_train)
pred = logreg.predict(Xc_test)
print(f"Accuracy: {accuracy_score(yc_test, pred):.3f}")
print("Wahrscheinlichkeiten der ersten 5 Testproben (Klasse 0, Klasse 1):")
print(logreg.predict_proba(Xc_test)[:5].round(3))

# Grösste Gewichte (auf skalierten Merkmalen, darum vergleichbar)
gewichte = pd.Series(logreg[-1].coef_[0], index=Xc.columns)
gewichte.reindex(gewichte.abs().sort_values(ascending=False).index).head(5)

In [ ]:
from sklearn.dummy import DummyClassifier

# Naive baseline: always predict the most frequent class
basis = DummyClassifier(strategy="most_frequent").fit(Xc_train, yc_train)
print(f"Baseline (immer die häufigste Klasse): Accuracy {basis.score(Xc_test, yc_test):.3f}")
print(f"Logistische Regression:                Accuracy {accuracy_score(yc_test, pred):.3f} ({len(yc_test)} Testproben)")

Die Baseline liegt bei etwa 63 %, dem Anteil der häufigsten Klasse. Ein Modell muss klar darüber liegen, sonst hat es nichts gelernt.

### Den Threshold verschieben

Ein tieferer Threshold sagt öfter «Klasse 1»: Recall steigt, Precision sinkt. Welcher Threshold richtig ist, hängt davon ab, welcher Fehler teurer ist.

In [ ]:
proba = logreg.predict_proba(Xc_test)[:, 1]
zeilen = []
for s in [0.2, 0.5, 0.8, 0.95]:
    p = (proba >= s).astype(int)
    zeilen.append({"Threshold": s, "Precision": precision_score(yc_test, p), "Recall": recall_score(yc_test, p),
                   "F1": f1_score(yc_test, p)})
pd.DataFrame(zeilen)

## Mehr als zwei Klassen

Drei Weinsorten (`load_wine`). Die logistische Regression arbeitet automatisch mit Softmax über alle Klassen. Die Confusion Matrix ist dann 3 × 3.

In [ ]:
Xw, yw = load_wine(return_X_y=True, as_frame=True)
Xw_train, Xw_test, yw_train, yw_test = train_test_split(Xw, yw, test_size=0.3, stratify=yw, random_state=SEED)
multi = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xw_train, yw_train)
pw = multi.predict(Xw_test)
print(confusion_matrix(yw_test, pw))
print(classification_report(yw_test, pw, digits=3))

## kNN (k-nächste Nachbarn)

Wirkung von `k` und warum Skalieren bei kNN Pflicht ist: Ohne Skalierung dominieren Merkmale mit grossen Zahlen (z. B. `mean area` in Hunderten) den Abstand.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

zeilen = []
for k in [1, 3, 5, 15, 51, 151]:
    roh = cross_val_score(KNeighborsClassifier(k), Xc_train, yc_train, cv=5).mean()
    skal = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(k)), Xc_train, yc_train, cv=5).mean()
    zeilen.append({"k": k, "ohne Skalieren": roh, "mit Skalieren": skal})
pd.DataFrame(zeilen)

**Deutung:** Mit Skalierung ist kNN deutlich besser. `k = 1` folgt jedem Trainingspunkt (Overfitting), ein sehr grosses `k` wird zu grob (Underfitting).

## Naive Bayes

Sehr schnell und ohne Hyperparameter. Oft etwas schwächer als die logistische Regression, aber eine gute erste Baseline.

In [ ]:
from sklearn.naive_bayes import GaussianNB

nb_cv = cross_val_score(GaussianNB(), Xc_train, yc_train, cv=5).mean()
lr_cv = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), Xc_train, yc_train, cv=5).mean()
print(f"Naive Bayes:           {nb_cv:.3f}")
print(f"Logistische Regression: {lr_cv:.3f}")

## SVM (Support Vector Machine)

Zwei ineinander greifende Halbmonde (`make_moons`). Eine lineare Grenze schafft das nicht, der RBF-Kernel schon. `C` steuert, wie streng Fehler bestraft werden.

In [ ]:
from sklearn.svm import SVC

Xm, ym = make_moons(n_samples=300, noise=0.25, random_state=SEED)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(Xm, ym, test_size=0.3, random_state=SEED)


def grenze(ax, model, X, y, titel):
    """Draw the decision regions of a fitted 2D classifier."""
    xx, yy = np.meshgrid(np.linspace(X[:, 0].min() - .5, X[:, 0].max() + .5, 200),
                         np.linspace(X[:, 1].min() - .5, X[:, 1].max() + .5, 200))
    zz = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=.25, cmap="coolwarm")
    ax.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="coolwarm")
    ax.set_title(titel, fontsize=10); ax.set_xticks([]); ax.set_yticks([])


fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, (name, svm) in zip(axes, [("linear", SVC(kernel="linear")), ("RBF, C=1", SVC(kernel="rbf", C=1)),
                                  ("RBF, C=1000", SVC(kernel="rbf", C=1000))]):
    m = make_pipeline(StandardScaler(), svm).fit(Xm_train, ym_train)
    print(f"{name:12s} Train {m.score(Xm_train, ym_train):.2f} · Test {m.score(Xm_test, ym_test):.2f}")
    grenze(ax, m, Xm_train, ym_train, f"{name}: Test {m.score(Xm_test, ym_test):.2f}")
plt.show()

**Deutung:** Die lineare Grenze ist eine Gerade und schneidet durch beide Halbmonde (Test etwa 0,88). RBF folgt der Krümmung und ist besser. Ein grosses `C` bestraft Fehler strenger, die Grenze schmiegt sich enger an die Trainingspunkte. Hier hilft das noch leicht. Bei mehr Rauschen oder weniger Daten kippt es ins Overfitting, darum wählt man `C` mit Cross-Validation.

## Entscheidungsbaum

Tiefe gegen Genauigkeit: Ein tiefer Baum lernt die Trainingsdaten auswendig. Ein flacher Baum lässt sich als Regeln lesen.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

zeilen = []
for d in [1, 2, 3, 5, 8, None]:
    baum = DecisionTreeClassifier(max_depth=d, random_state=SEED).fit(Xc_train, yc_train)
    zeilen.append({"max_depth": d, "Train": baum.score(Xc_train, yc_train), "Test": baum.score(Xc_test, yc_test)})
print(pd.DataFrame(zeilen))

flach = DecisionTreeClassifier(max_depth=2, random_state=SEED).fit(Xc_train, yc_train)
print(export_text(flach, feature_names=list(Xc.columns)))

## Unausgewogene Klassen

5 % Positive. «Immer negativ» hätte 95 % Accuracy. Verglichen werden ein normales Modell, `class_weight="balanced"` und SMOTE (aus `imblearn`, nur auf den Trainingsdaten).

In [ ]:
Xi, yi = make_classification(n_samples=3000, n_features=8, n_informative=4, weights=[0.95, 0.05],
                             flip_y=0.01, random_state=SEED)
Xi_train, Xi_test, yi_train, yi_test = train_test_split(Xi, yi, test_size=0.3, stratify=yi, random_state=SEED)

modelle = {
    "normal": make_pipeline(StandardScaler(), LogisticRegression()),
    "class_weight=balanced": make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced")),
}
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import make_pipeline as make_imb_pipeline
    # imblearn's pipeline applies SMOTE only during fit, i.e. only to the training data
    modelle["SMOTE"] = make_imb_pipeline(StandardScaler(), SMOTE(random_state=SEED), LogisticRegression())
except ImportError:
    print("imblearn fehlt: pip install imbalanced-learn")

zeilen = []
for name, m in modelle.items():
    p = m.fit(Xi_train, yi_train).predict(Xi_test)
    zeilen.append({"Modell": name, "Accuracy": accuracy_score(yi_test, p), "Precision": precision_score(yi_test, p),
                   "Recall": recall_score(yi_test, p), "F1": f1_score(yi_test, p)})
pd.DataFrame(zeilen)

**Deutung:** Das normale Modell hat 95 % Accuracy, findet aber fast keinen Positiven (Recall um 0,04). Das ist kaum besser als «immer negativ». Gewichtung und SMOTE finden rund 45 % der Positiven, bezahlen das aber mit viel falschem Alarm (Precision unter 0,1). Die Daten sind absichtlich schwer trennbar. Welche Balance richtig ist, hängt davon ab, was ein verpasster Fall und was ein falscher Alarm kostet. Nachsteuern lässt sich mit dem Threshold.